# Build a RAG App Over a GitHub Repo — Colab / Kaggle / Binder companion

This notebook mirrors the local `uv` project from the course's
[Build a RAG App Over a GitHub Repo](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/github-repo-rag)
lesson, adapted to run in a hosted notebook: it clones the course repo itself, chunks its
lesson docs (code-style and prose-style chunking), embeds them locally with
`sentence-transformers`, retrieves with NumPy cosine similarity, and answers with
file-and-line citations via a free-tier LLM.

See the [lesson](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/github-repo-rag) for the full walkthrough and the
[local example project](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/github-repo-rag) for the real, file-based version of this same code.

## Step 0: Install dependencies

In [ ]:
!pip install sentence-transformers numpy openai

## Step 1: Clone a test repo

A hosted notebook has no local repo to read from, so clone one here. The course repo
itself makes a great test subject — small, and full of lessons you'll recognize. We use
`--depth 1` to skip git history, and index only its `docs/projects/` folder so embedding
stays fast.

In [ ]:
!git clone --depth 1 https://github.com/abderrahim-lectures/python-data-analysis-course
REPO_ROOT = "python-data-analysis-course"
INDEX_ROOT = f"{REPO_ROOT}/docs/projects"
print("Cloned.")

## Step 2: Walk the repo and chunk its files

Same logic as [`prepare_repo.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/github-repo-rag/prepare_repo.py)
in the local example: code files are split at top-level `def ` / `class ` boundaries,
prose (Markdown) files by paragraph, and every chunk records its source file **and line
range** — that metadata is what lets the final step produce verifiable citations.

In [ ]:
from pathlib import Path

SKIP_DIRS = {".git", "node_modules", "__pycache__", "venv", ".venv", "dist", "build"}
CODE_EXTS = {".py", ".js", ".ts", ".go", ".rs", ".java", ".md", ".mdx", ".txt"}
TARGET_CHUNK_SIZE = 500  # characters


def walk_repo(repo_root: Path) -> list[Path]:
    files = []
    for path in sorted(repo_root.rglob("*")):
        if path.is_dir():
            continue
        if any(part in SKIP_DIRS for part in path.relative_to(repo_root).parts):
            continue
        if path.suffix.lower() in CODE_EXTS:
            files.append(path)
    return files


def chunk_code(text: str, source: str) -> list[dict]:
    lines = text.splitlines()
    boundaries = [0]
    for i, line in enumerate(lines):
        if line.startswith(("def ", "class ")) and not line.startswith(("    ", "\t")):
            if i > 0:
                boundaries.append(i)
    boundaries.append(len(lines))
    chunks = []
    for start, end in zip(boundaries, boundaries[1:]):
        body = "\n".join(lines[start:end]).strip()
        if not body:
            continue
        chunks.append({"source": source, "start": start + 1, "end": end, "text": body})
    return chunks


def chunk_prose(text: str, source: str) -> list[dict]:
    lines = text.splitlines()
    chunks = []
    current, start = [], None
    for i, line in enumerate(lines):
        if not line.strip():
            if current:
                chunks.append({"source": source, "start": start + 1, "end": i,
                               "text": "\n".join(current).strip()})
                current, start = [], None
            continue
        if start is None:
            start = i
        current.append(line)
        if sum(len(l) for l in current) >= TARGET_CHUNK_SIZE:
            chunks.append({"source": source, "start": start + 1, "end": i + 1,
                           "text": "\n".join(current).strip()})
            current, start = [], None
    if current:
        chunks.append({"source": source, "start": start + 1, "end": len(lines),
                       "text": "\n".join(current).strip()})
    return chunks


def load_chunks(repo_root: Path) -> list[dict]:
    chunks = []
    for path in walk_repo(repo_root):
        text = path.read_text(encoding="utf-8", errors="replace")
        source = str(path.relative_to(repo_root))
        if path.suffix.lower() in {".md", ".mdx", ".txt"}:
            chunks.extend(chunk_prose(text, source))
        else:
            chunks.extend(chunk_code(text, source))
    return chunks


chunks = load_chunks(Path(INDEX_ROOT))
print(f"Indexed {len(chunks)} chunks from {len(walk_repo(Path(INDEX_ROOT)))} files")
for c in chunks[:3]:
    print(f"  {c['source']}:{c['start']}-{c['end']}  {c['text'][:60]}...")

## Step 3: Embed the chunks locally

Same model as [`build_index.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/github-repo-rag/build_index.py):
`all-MiniLM-L6-v2`, run entirely on CPU, no API key needed. Since this notebook has no
`index.npy`/`chunks.json` files to persist to, the embeddings are kept in memory.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Embedding {len(chunks)} chunks with {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, normalize_embeddings=True)

print(f"Embedded {embeddings.shape[0]} chunks ({embeddings.shape[1]}-dim)")

## Step 4: Retrieve relevant chunks

Same cosine-similarity retrieval as [`retrieve.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/github-repo-rag/retrieve.py) —
every embedding is already normalized to length 1, so cosine similarity collapses to a
plain dot product. Results now carry the line range too, ready for citation.

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[dict]:
    question_vector = model.encode([question], normalize_embeddings=True)[0]
    similarities = embeddings @ question_vector
    top_indices = np.argsort(similarities)[::-1][:top_k]
    return [
        {**chunks[i], "score": float(similarities[i])}
        for i in top_indices
    ]


for r in retrieve("How does the course build this website?"):
    print(f"{r['score']:.3f}  [{r['source']}:{r['start']}-{r['end']}]  {r['text'][:70]}...")

## Step 5: Get a free-tier LLM API key

Pick any provider from the table in the [lesson's Setup](https://abderrahim-lectures.github.io/python-data-analysis-course/docs/projects/github-repo-rag#get-a-free-llm-api-key) —
GitHub Models is the suggested default since it needs no separate signup. Paste the key
below; `getpass` keeps it out of the notebook's saved output and cell history.

In [ ]:
import getpass
import os

os.environ["GITHUB_TOKEN"] = getpass.getpass("Paste your GitHub Models API key: ")

## Step 6: Generate an answer with citations

Same prompt and client setup as [`query.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/blob/main/examples/github-repo-rag/query.py) —
GitHub Models exposes an OpenAI-compatible API, so the plain `openai` client library works
without any extra package. The prompt demands `[path:start-end]` citations, so the answer
points you at the exact lines to open and verify.

In [ ]:
from openai import OpenAI

PROMPT_TEMPLATE = """Answer the question using ONLY the context below. The
context is drawn from a real code repository; each block is tagged with the
file and line range it came from. Support every factual claim you make with a
citation in the form [path.py:start-end]. If the context doesn't contain the
answer, say so -- do not make something up, and do not cite anything not in
the context.

Context:
{context}

Question: {question}

Answer:"""


def build_prompt(question: str, retrieved: list[dict]) -> str:
    context = "\n\n".join(
        f"[{c['source']}:{c['start']}-{c['end']}]\n{c['text']}" for c in retrieved
    )
    return PROMPT_TEMPLATE.format(context=context, question=question)


def ask(question: str, top_k: int = 4) -> str:
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)

    client = OpenAI(
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference",
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # confirm this still has a free tier before running
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

## Try it: ask a question about the course repo

In [ ]:
question = "How does the course build this website?"
answer = ask(question)
print(f"Q: {question}\n")
print(f"A: {answer}")